# Document OCR with MiniCPM-OCR and OpenVINO

MiniCPM-OCR is a document-understanding model from the [MiniCPM-V](https://huggingface.co/papers/2509.18154) family developed by [OpenBMB](https://github.com/OpenBMB). It keeps the [MiniCPM-V 4.6](../minicpm-v-4.6/minicpm-v-4.6.ipynb) architecture, a SigLIP vision encoder with a window-attention merger in front of a Qwen3.5 language model, and specializes it for reading text-dense images: text, tables, formulas, charts, seals and key information extraction. See the [model card](https://huggingface.co/openbmb/MiniCPM-OCR-1.0), the [MiniCPM-OCR repository](https://github.com/OpenBMB/MiniCPM-OCR) and the [Transformers documentation](https://huggingface.co/docs/transformers/main/model_doc/minicpm_ocr) for details.

In this tutorial we export [`openbmb/MiniCPM-OCR-1.0`](https://huggingface.co/openbmb/MiniCPM-OCR-1.0) to OpenVINO IR with [Optimum Intel](https://github.com/huggingface/optimum-intel), compress its weights with [NNCF](https://github.com/openvinotoolkit/nncf), and run inference with `OVModelForVisualCausalLM`: we recognize the text of a receipt, convert a form to Markdown, read Chinese text from an image crop and extract key information from a document. Then we check OCR results with [MiniCPM-OCR Jev](https://huggingface.co/openbmb/MiniCPM-OCR-Jev), an OCR quality-decision model that scores whether a candidate transcription matches the image. Finally, we launch an interactive Gradio demo.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert and compress the model](#Convert-and-compress-the-model)
- [Load the model with OpenVINO](#Load-the-model-with-OpenVINO)
- [Run OCR](#Run-OCR)
    - [Recognize the text of a receipt](#Recognize-the-text-of-a-receipt)
    - [Convert a page to Markdown](#Convert-a-page-to-Markdown)
    - [Read multilingual text](#Read-multilingual-text)
    - [Extract key information](#Extract-key-information)
- [Verify OCR results with MiniCPM-OCR Jev](#Verify-OCR-results-with-MiniCPM-OCR-Jev)
- [Measure accuracy and performance](#Measure-accuracy-and-performance)
- [Interactive demo](#Interactive-demo)

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO and uses development builds of Optimum Intel and OpenVINO. It may be fully supported and validated in the future.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/minicpm-ocr/minicpm-ocr.ipynb" />

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

MiniCPM-OCR and MiniCPM-OCR Jev export is available in the main branch of Optimum Intel, and the model itself is available in Transformers starting from version 5.21.0. Optimum Intel declares an older Transformers requirement, so Transformers is installed after Optimum Intel. OpenVINO and OpenVINO Tokenizers are installed from nightly builds. Restart the kernel after installation if any of these packages were already imported.

In [ ]:
%pip uninstall -q -y optimum optimum-intel optimum-onnx
%pip install -q "git+https://github.com/huggingface/optimum-intel.git" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "transformers==5.21.0" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "torch>=2.4" torchvision nncf datasets Pillow opencv-python requests ipywidgets huggingface_hub "gradio>=5,<7" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q --pre -U openvino openvino-tokenizers --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly

In [ ]:
from pathlib import Path

import requests

for name in ("cmd_helper.py", "notebook_utils.py"):
    if not Path(name).exists():
        response = requests.get(
            f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{name}",
            timeout=30,
        )
        response.raise_for_status()
        Path(name).write_text(response.text, encoding="utf-8")

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("minicpm-ocr.ipynb")

## Convert and compress the model
[back to top ⬆️](#Table-of-contents:)

MiniCPM-OCR is a PyTorch model. We convert it to OpenVINO Intermediate Representation (IR) with the `optimum-cli` tool of [Optimum Intel](https://huggingface.co/docs/optimum/intel/index):

```bash
optimum-cli export openvino --model <model_id_or_path> --task image-text-to-text --weight-format <format> <output_dir>
```

The model is exported as three OpenVINO models: the vision encoder, the text embeddings and the Qwen3.5 language model. Weight compression with [NNCF](https://github.com/openvinotoolkit/nncf) reduces their memory footprint. OCR needs character-exact output, so the weight format affects quality more than for a conversational model:

* `fp16` stays closest to the original model and needs the most memory.
* `int8` (default) is a balance between size and quality.
* `int4` applies the data-aware recipe that Optimum Intel defines for this checkpoint (AWQ and scale estimation calibrated on TextVQA samples). It is the smallest and fastest option, takes longer to export and downloads the calibration dataset.
* `int4 (NPU)` is the recipe for running the language model on NPU: symmetric int4 with group size 128 for all layers (`--sym --group-size 128 --ratio 1 --backup-precision none`) with AWQ and scale estimation on TextVQA.

Accuracy and speed depend on the weight format, the device and the software versions; the [Measure accuracy and performance](#Measure-accuracy-and-performance) section measures them for your setup.

`Model:` accepts a Hugging Face model ID or a local checkpoint directory.

In [ ]:
import ipywidgets as widgets

model_source = widgets.Text(value="openbmb/MiniCPM-OCR-1.0", description="Model:")
weight_format = widgets.Dropdown(options=["int8", "int4", "int4 (NPU)", "fp16"], value="int8", description="Weights:")
display(model_source, weight_format)

In [ ]:
import os
import sys

from cmd_helper import optimum_cli

EXPORT_ARGS = {
    "int8": {"weight-format": "int8"},
    "int4": {"weight-format": "int4"},
    "int4 (NPU)": {
        "weight-format": "int4",
        "sym": "",
        "group-size": 128,
        "ratio": 1,
        "awq": "",
        "scale-estimation": "",
        "backup-precision": "none",
        "dataset": "textvqa",
    },
    "fp16": {"weight-format": "fp16"},
}

model_id = model_source.value.strip()
model_dir = Path(f"{Path(model_id).name}-ov") / weight_format.value.replace(" (NPU)", "-npu")

# make `optimum-cli` of the notebook environment available to the export subprocess
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]
if not (model_dir / "openvino_language_model.xml").exists():
    optimum_cli(model_id, model_dir, additional_args={"task": "image-text-to-text", **EXPORT_ARGS[weight_format.value]})
print(f"OpenVINO model: {model_dir}")

## Load the model with OpenVINO
[back to top ⬆️](#Table-of-contents:)

`OVModelForVisualCausalLM` loads the three OpenVINO models and provides the familiar Transformers `generate` API, while the Transformers processor prepares image slices and the chat prompt.

GPU runs in f16 inference precision by default, which can change single characters of a transcription. Select **Full precision on GPU** to request f32 inference precision at the cost of speed.

**NPU (preview).** The language model can run on NPU through [NPUW](https://docs.openvino.ai/2025/openvino-workflow-generative/inference-with-genai/inference-with-genai-on-npu.html), which compiles it with static shapes; the vision encoder and the text embeddings stay on CPU. Use the `int4 (NPU)` weights for it. The prompt of a document image in `4x` mode takes up to a few thousand tokens, so the maximum prompt length is set to 4096 tokens. Chunked prefill is disabled (`NPUW_LLM_PREFILL_CHUNK_SIZE: 0`) until the fix for Qwen3.5 linear attention ([openvino#39125](https://github.com/openvinotoolkit/openvino/pull/39125)) is available in nightly builds.

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="AUTO")
gpu_full_precision = widgets.Checkbox(value=False, description="Full precision on GPU")
display(device, gpu_full_precision)

In [ ]:
import openvino as ov
from optimum.intel import OVModelForVisualCausalLM
from transformers import AutoProcessor

NPUW_CONFIG = {
    "NPU_USE_NPUW": "YES",
    "NPUW_LLM": "YES",
    "NPUW_LLM_MAX_PROMPT_LEN": 4096,
    "NPUW_LLM_MIN_RESPONSE_LEN": 1024,
    "NPUW_LLM_PREFILL_CHUNK_SIZE": 0,
}

use_npu = device.value == "NPU"
if use_npu and not weight_format.value.endswith("(NPU)"):
    print("Warning: select the `int4 (NPU)` weights for NPU, other weight formats may fail to compile or lose accuracy.")
ov_config = {"INFERENCE_PRECISION_HINT": "f32"} if gpu_full_precision.value and "GPU" in device.value else None

processor = AutoProcessor.from_pretrained(model_dir)
ov_model = OVModelForVisualCausalLM.from_pretrained(model_dir, device="CPU" if use_npu else device.value, ov_config=ov_config, compile=not use_npu)
if use_npu:
    ov_model.language_model.request = ov.Core().compile_model(ov_model.language_model.model, "NPU", NPUW_CONFIG).create_infer_request()
    ov_model.compile()
print(f"Model loaded on {device.value}, visual downsampling: {ov_model.config.downsample_mode}")

## Run OCR
[back to top ⬆️](#Table-of-contents:)

The processor applies the model chat template, slices the image and inserts the visual placeholders. A few details are specific to MiniCPM-OCR:

* **Prompts.** The prompt selects the task and should match the image region, as listed in the [model card](https://huggingface.co/openbmb/MiniCPM-OCR-1.0#prompts): `Text Recognition:` for text blocks, paragraphs, titles and pages, `Table Recognition:` for tables (HTML output), `Formula Recognition:` for formulas (LaTeX output), `Seal Recognition:` for seals and `Chart to Table:` for charts. The full parsing pipeline of MiniCPM-OCR detects regions with a layout model first and recognizes each crop; in this notebook we pass whole images and crops directly.

* **Visual downsampling.** The `4x` mode (default) keeps 4× more visual tokens than `16x`, which is what document and screenshot transcription needs. The OpenVINO vision encoder is exported for the mode from the model config, so the processor must use the same mode; the `ocr` function below passes it explicitly.
* **Thinking.** The chat template opens a thinking block unless `enable_thinking=False` is passed. Plain transcription does not need it, so it is disabled by default.
* **Small images.** Images and slices smaller than 448×448 pixels are upscaled by default (`small_image_upsize="always"`), which is the setting the checkpoint was trained with. Screenshots and crops are sometimes read better at their native resolution, where upscaling only blurs the glyphs: `"never"` keeps the native resolution, and `"adaptive"` decides per image and per slice from glyph size and density (requires OpenCV). This is a processor setting, so it does not require exporting the model again.
* **Greedy decoding** (`do_sample=False`) keeps transcriptions deterministic.

In [ ]:
from io import BytesIO

from PIL import Image
from transformers import TextStreamer


def load_image(url, path):
    path = Path(path)
    if not path.exists():
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        path.write_bytes(response.content)
    return Image.open(path).convert("RGB")


def ocr(image, prompt, *, enable_thinking=False, max_new_tokens=512, small_image_upsize="always"):
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=enable_thinking,
        processor_kwargs={"downsample_mode": ov_model.config.downsample_mode, "small_image_upsize": small_image_upsize},
    )
    streamer = TextStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True)
    output_ids = ov_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, streamer=streamer)
    return processor.decode(output_ids[0, inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()

### Recognize the text of a receipt
[back to top ⬆️](#Table-of-contents:)

In [ ]:
receipt = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/model_doc/ocr_receipt.jpeg",
    "ocr_receipt.jpeg",
)
display(receipt.resize((receipt.width * 512 // receipt.height, 512)))
receipt_text = ocr(receipt, "Text Recognition:")

### Convert a page to Markdown
[back to top ⬆️](#Table-of-contents:)

The model can also restore the document structure, such as headings, fields and lists, in Markdown.

In [ ]:
form = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/model_doc/ocr_suggestion_form.jpg",
    "ocr_suggestion_form.jpg",
)
display(form.resize((form.width * 640 // form.height, 640)))
form_markdown = ocr(form, "Convert this page to markdown.", max_new_tokens=1024)

### Read multilingual text
[back to top ⬆️](#Table-of-contents:)

Text is often read from regions cut out by a layout analysis step. Here we crop a Chinese caption from the example image of the [PaddleOCR Space](https://huggingface.co/spaces/PaddlePaddle/PaddleOCR).

In [ ]:
comic = load_image("https://huggingface.co/spaces/PaddlePaddle/PaddleOCR/resolve/main/cn_example.jpg", "cn_example.jpg")
caption = comic.crop((700, 525, 1135, 820))
display(caption)
caption_text = ocr(caption, "Text Recognition:", max_new_tokens=128)

### Extract key information
[back to top ⬆️](#Table-of-contents:)

MiniCPM-OCR can extract specified fields straight from an image as a JSON object, without producing the page text first. Replace the field names in the prompt with your own; the answer is generated text, so parse it and check the values before using them downstream.

In [ ]:
KIE_PROMPT = 'Extract the {fields}. Format the output as a JSON object with keys: {fields}. If an item is not in the image, use "###".'

kie_prompt = KIE_PROMPT.format(fields='"document_no", "date", "cashier", "total"')
answer = ocr(receipt, kie_prompt, max_new_tokens=128)

## Verify OCR results with MiniCPM-OCR Jev
[back to top ⬆️](#Table-of-contents:)

[MiniCPM-OCR Jev](https://huggingface.co/openbmb/MiniCPM-OCR-Jev) is a 2B OCR quality-decision model trained from Qwen3.5-2B, with the language model head replaced by a binary classification head. It receives the original region image, the candidate OCR text, an image of the text rendered with a Chinese font, and a task description in Chinese, and returns a score from 0 to 1: the higher, the more likely the candidate is correct. Use it to auto-accept reliable regions, route the rest to human review, or pick the best result among several OCR models.

Jev judges text, formula and table regions as they are cut out by a layout analysis step rather than full pages. There is no universal decision threshold: the model card recommends choosing it on labeled samples of your documents. Here we use 0.5606, the threshold of the Optimum Intel example. We use it to check the transcription of the Chinese caption from the [Read multilingual text](#Read-multilingual-text) example and the same text with one character changed.

Jev is exported with the `text-classification` task and loaded with `OVModelForVisualSequenceClassification`:

```bash
optimum-cli export openvino --model <jev_model_id_or_path> --task text-classification --weight-format <format> <output_dir>
```

`Jev model:` accepts a Hugging Face model ID or a local checkpoint directory. Scores are compared against a threshold, so `fp16` weights, closest to the original model, are used by default. NPU is not supported for Jev yet.

In [ ]:
jev_source = widgets.Text(value="openbmb/MiniCPM-OCR-Jev", description="Jev model:")
jev_weight_format = widgets.Dropdown(options=["fp16", "int8"], value="fp16", description="Weights:")
jev_device = device_widget(default="AUTO", exclude=["NPU"])
display(jev_source, jev_weight_format, jev_device)

In [ ]:
from optimum.intel import OVModelForVisualSequenceClassification

jev_id = jev_source.value.strip()
jev_dir = Path(f"{Path(jev_id).name}-ov") / jev_weight_format.value

if not (jev_dir / "openvino_language_model.xml").exists():
    optimum_cli(jev_id, jev_dir, additional_args={"task": "text-classification", "weight-format": jev_weight_format.value})

jev_processor = AutoProcessor.from_pretrained(jev_dir)
jev_model = OVModelForVisualSequenceClassification.from_pretrained(jev_dir, device=jev_device.value)
print(f"Jev loaded on {jev_device.value}")

In [ ]:
from huggingface_hub import hf_hub_download
from PIL import ImageDraw, ImageFont

JEV_THRESHOLD = 0.5606135129928589
JEV_TASK = "第一张图是原图，第二张图是上述 OCR 文本渲染得到的图片。\n请判断该 OCR 文本及其对应渲染图是否与原图内容一致。\n这是一个二分类任务：一致标为 1，不一致标为 0。"

# the Chinese font the PaddleOCR Space draws its results with
font = ImageFont.truetype(hf_hub_download("PaddlePaddle/PaddleOCR", "simfang.ttf", repo_type="space"), 32)


def render_text(text):
    left, top, right, bottom = ImageDraw.Draw(Image.new("RGB", (1, 1))).multiline_textbbox((0, 0), text, font=font)
    render = Image.new("RGB", (right - left + 32, bottom - top + 32), "white")
    ImageDraw.Draw(render).multiline_text((16 - left, 16 - top), text, font=font, fill="black")
    return render


def jev_score(image, text):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": f"OCR文本：\n{text}\n"},
                {"type": "image", "image": render_text(text)},
                {"type": "text", "text": JEV_TASK},
            ],
        }
    ]
    # no generation prompt: the classified token is the newline after <|im_end|>
    inputs = jev_processor.apply_chat_template(messages, add_generation_prompt=False, tokenize=True, return_dict=True, return_tensors="pt")
    return jev_model(**inputs).logits.softmax(-1)[0, 1].item()


def jev_verdict(image, text):
    probability = jev_score(image, text)
    return f"P(consistent) = {probability:.4f} -> {'consistent' if probability >= JEV_THRESHOLD else 'inconsistent'}"

In [ ]:
def change_one_character(text):
    if "四蹄" in text:
        return text.replace("四蹄", "三蹄")  # four hooves -> three hooves
    positions = [i for i, char in enumerate(text) if not char.isspace()]
    if not positions:
        return text + "三"
    i = positions[len(positions) // 2]
    return text[:i] + ("三" if text[i] != "三" else "四") + text[i + 1 :]


display(render_text(caption_text))
for name, text in {"MiniCPM-OCR": caption_text, "one character changed": change_one_character(caption_text)}.items():
    print(f"{name}: {text!r}")
    print(f"    {jev_verdict(caption, text)}")

## Measure accuracy and performance
[back to top ⬆️](#Table-of-contents:)

Weight compression and reduced inference precision can change characters of a transcription. Here we run the examples above with the OpenVINO model and with the original PyTorch model and compare:

* **Similarity**: character-level similarity of the OpenVINO and PyTorch transcriptions (`difflib.SequenceMatcher` ratio, 1.0 means identical text).
* **Identical answers**: the number of examples with exactly the same output.
* **Time to first token**: image encoding and prompt processing of the OpenVINO model.
* **Throughput**: generated tokens per second after the first token.

The PyTorch model is loaded in FP32 on CPU, which needs about 5 GB of RAM for MiniCPM-OCR-1.0. Clear the checkbox to measure only the performance of the OpenVINO model. Answers are limited to `max_new_tokens=256` to keep the measurement short.

In [ ]:
compare_with_pytorch = widgets.Checkbox(value=True, description="Compare with the original PyTorch model")
compare_with_pytorch

In [ ]:
import gc
import time
from difflib import SequenceMatcher

import torch
from transformers import AutoModelForImageTextToText
from transformers.generation.streamers import BaseStreamer


class TimingStreamer(BaseStreamer):
    # `generate` puts the prompt first, then every generated token.
    def __init__(self):
        self.times = []

    def put(self, value):
        self.times.append(time.perf_counter())

    def end(self):
        pass


def transcribe(model, image, prompt, max_new_tokens=256, streamer=None):
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
        processor_kwargs={"downsample_mode": ov_model.config.downsample_mode},
    )
    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, streamer=streamer)
    return processor.decode(output_ids[0, inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()


samples = {
    "receipt": (receipt, "Text Recognition:"),
    "form": (form, "Convert this page to markdown."),
    "caption": (caption, "Text Recognition:"),
    "kie": (receipt, kie_prompt),
}

transcribe(ov_model, caption, "Text Recognition:", max_new_tokens=8)  # warm-up

ov_results = {}
for name, (image, prompt) in samples.items():
    streamer = TimingStreamer()
    text = transcribe(ov_model, image, prompt, streamer=streamer)
    first_token, last_token = streamer.times[1], streamer.times[-1]
    new_tokens = len(streamer.times) - 2
    ov_results[name] = {
        "text": text,
        "ttft": first_token - streamer.times[0],
        "throughput": new_tokens / (last_token - first_token) if new_tokens else float("nan"),
    }

if compare_with_pytorch.value:
    pt_model = AutoModelForImageTextToText.from_pretrained(model_id, dtype=torch.float32).eval()
    with torch.no_grad():
        for name, (image, prompt) in samples.items():
            ov_results[name]["similarity"] = SequenceMatcher(None, transcribe(pt_model, image, prompt), ov_results[name]["text"]).ratio()
    del pt_model
    gc.collect()

print(f"{'example':<10} {'similarity':>10} {'TTFT, s':>8} {'tokens/s':>9}")
for name, result in ov_results.items():
    similarity = f"{result['similarity']:.4f}" if "similarity" in result else "-"
    print(f"{name:<10} {similarity:>10} {result['ttft']:>8.2f} {result['throughput']:>9.1f}")
if compare_with_pytorch.value:
    similarities = [result["similarity"] for result in ov_results.values()]
    print(f"\nMean similarity: {sum(similarities) / len(similarities):.4f}")
    print(f"Identical answers: {sum(similarity == 1.0 for similarity in similarities)}/{len(similarities)}")
print(f"Device: {device.value}, weights: {model_dir}")

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

Upload an image or select an example, choose a prompt and run OCR. The answer is streamed as it is generated. With **Verify with Jev** selected, the result is scored with MiniCPM-OCR Jev when generation is finished; use crops of text, formula or table regions for meaningful scores.

In [ ]:
if not Path("gradio_helper.py").exists():
    response = requests.get(
        "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/minicpm-ocr/gradio_helper.py",
        timeout=30,
    )
    response.raise_for_status()
    Path("gradio_helper.py").write_text(response.text, encoding="utf-8")

from gradio_helper import make_demo

caption.save("cn_caption.png")
demo = make_demo(ov_model, processor, examples=["ocr_receipt.jpeg", "ocr_suggestion_form.jpg", "cn_caption.png"], verify=jev_verdict)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)
# If you are launching remotely, specify server_name and server_port
# EXAMPLE: `demo.launch(server_name='your server name', server_port='server port in int')`
# To learn more please refer to the Gradio docs: https://gradio.app/docs/